# Importing Necessary Functions

In [2]:
from skimage.filters import threshold_local
import numpy as np
import argparse
import cv2
import imutils
import tkinter as tk
from tkinter import filedialog

# Defining functions necessary for scanning

In [3]:
def order_points(pts):
	# initialize a list of coordinates that will be ordered
	rect = np.zeros((4, 2), dtype = "float32")
	# the top-left point will have the smallest sum, whereas
	# the bottom-right point will have the largest sum
	s = pts.sum(axis = 1)
	rect[0] = pts[np.argmin(s)]
	rect[2] = pts[np.argmax(s)]
	# now, compute the difference between the points, the
	# top-right point will have the smallest difference,
	# whereas the bottom-left will have the largest difference
	diff = np.diff(pts, axis = 1)
	rect[1] = pts[np.argmin(diff)]
	rect[3] = pts[np.argmax(diff)]
	# return the ordered coordinates
	return rect

In [4]:
def four_point_transform(image, pts):
	# obtain a consistent order of the points and unpack them
	# individually
	rect = order_points(pts)
	(tl, tr, br, bl) = rect
	# compute the width of the new image, which will be the
	# maximum distance between bottom-right and bottom-left
	# x-coordiates or the top-right and top-left x-coordinates
	width1 = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
	width2 = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
	maxWidth = max(int(width1), int(width2))
	# compute the height of the new image, which will be the
	# maximum distance between the top-right and bottom-right
	# y-coordinates or the top-left and bottom-left y-coordinates
	height1 = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
	height2 = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
	maxHeight = max(int(height1), int(height2))
	# now that we have the dimensions of the new image, construct
	# the set of destination points to obtain a "birds eye view",
	# (i.e. top-down view) of the image, again specifying points
	# in the top-left, top-right, bottom-right, and bottom-left
	# order
	dst = np.array([
		[0, 0],
		[maxWidth - 1, 0],
		[maxWidth - 1, maxHeight - 1],
		[0, maxHeight - 1]], dtype = "float32")
	# compute the perspective transform matrix and then apply it
	M = cv2.getPerspectiveTransform(rect, dst)
	warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
	# return the warped image
	return warped

# Parsing arguments

In [ ]:
ap = argparse.ArgumentParser()
ap.add_argument("-i", "--image", required=False, help="Path to the image to be scanned")
parsed_args, _ = ap.parse_known_args()
args = vars(parsed_args)

if not args.get("image"):
	root = tk.Tk()
	root.withdraw()
	root.attributes('-topmost', True)
	root.update()
	args["image"] = filedialog.askopenfilename(
		title="Select an image to scan",
		filetypes=[
			("Image Files", "*.jpg *.jpeg *.png *.bmp *.tiff"),
			("All Files", "*.*")
		]
	)
	root.destroy()
	if not args["image"]:
		raise RuntimeError("No image selected. Please provide an image path or choose one in the dialog.")

# Edge Detection

In [ ]:
image = cv2.imread(args['image'])
if image is None:
    raise FileNotFoundError(f"Unable to load image at {args['image']}")
ratio = image.shape[0] / 500.0
orig = image.copy()
image = imutils.resize(image, height=500)

# convert image to grayscale, blur, and find edges
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (5, 5), 0)
edged = cv2.Canny(gray, 50, 150)
# close small gaps and widen edges so the page boundary is more likely to be detected as a single contour
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
edged = cv2.morphologyEx(edged, cv2.MORPH_CLOSE, kernel)
edged = cv2.dilate(edged, kernel, iterations=2)

# original image vs edge detected image
cv2.imshow("Image", image)
cv2.imshow("Edged", edged)
cv2.waitKey()
cv2.destroyAllWindows()

# Finding contours in the image

In [ ]:
# find the contours in the edged image, keeping only the
# external contour so we target the receipt outline instead of background
cnts = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)
cnts = sorted(cnts, key = cv2.contourArea, reverse = True)
# choose the best contour by page-like shape and reasonable area
image_area = image.shape[0] * image.shape[1]
best_page = None
best_score = float('inf')
for c in cnts:
	area = cv2.contourArea(c)
	if area < image_area * 0.05:
		continue
	hull = cv2.convexHull(c)
	peri = cv2.arcLength(hull, True)
	approx = cv2.approxPolyDP(hull, 0.01 * peri, True)
	if len(approx) == 4:
		x, y, w, h = cv2.boundingRect(approx)
		if h == 0:
			continue
		aspect = float(w) / h
		area_ratio = area / image_area
		aspect_score = abs(aspect - 0.35)
		area_score = abs(area_ratio - 0.20)
		if 0.15 < aspect < 0.7 and area_ratio > 0.08:
			score = aspect_score + area_score
			if score < best_score:
				best_score = score
				best_page = approx
# if we didn't find a good 4-point contour, use a rotated rectangle around the largest contour
if best_page is None and len(cnts) > 0:
	best = cnts[0]
	hull = cv2.convexHull(best)
	rect = cv2.minAreaRect(hull)
	box = cv2.boxPoints(rect)
	best_page = np.array(box, dtype="float32")

screenCnt = best_page
# show the contour (outline) of the piece of paper
print("STEP 2: Find contours of paper")
if screenCnt is not None and screenCnt.size >= 4:
	contour_to_draw = np.array(screenCnt, dtype=np.int32)
	if contour_to_draw.ndim == 2:
		contour_to_draw = contour_to_draw.reshape((-1, 1, 2))
	if contour_to_draw.size > 0 and contour_to_draw.shape[0] > 0:
		cv2.drawContours(image, [contour_to_draw], -1, (0, 255, 0), 2) # type: ignore
	else:
		x, y, w, h = 0, 0, image.shape[1], image.shape[0] # type: ignore
		cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2) # type: ignore
else:
	# fallback to a rectangle around the full frame if nothing else is found
	x, y, w, h = 0, 0, image.shape[1], image.shape[0]
	cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)
cv2.imshow("Outline", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

STEP 2: Find contours of paper


# Applying a perspective and threshold

In [ ]:
# apply the four point transform to obtain a top-down
# view of the original image
if screenCnt is None:
	raise RuntimeError("No 4-point contour found; cannot apply perspective transform")
pts = (screenCnt.reshape(4, 2) * ratio).astype("float32")
warped = four_point_transform(orig, pts)
# convert the warped image to grayscale, then threshold it
# to give it that 'black and white' paper effect
warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
T = threshold_local(warped, 11, offset = 10, method = "gaussian")
warped = (warped > T).astype("uint8") * 255
# show the original and scanned images
cv2.imshow("Scanned", imutils.resize(warped, height = 650))
cv2.waitKey(0)

-1

# Saving to Files

In [ ]:
root = tk.Tk()
root.withdraw()
save_path = filedialog.asksaveasfilename(
	title="Save scanned image as",
	defaultextension=".jpg",
	filetypes=[("JPEG Image", "*.jpg"), ("PNG Image", "*.png"), ("All Files", "*.*")],
	initialfile="scanned_output.jpg"
)
if save_path:
	cv2.imwrite(save_path, warped)
	print(f"Saved scanned image as: {save_path}")
else:
	print("Save canceled.")

Save canceled.
